# 03. PEFT / LoRA

## 학습 목표
- PEFT가 왜 필요한지 이해
- LoRA의 수학적 원리 (Low-Rank Decomposition) 이해
- QLoRA (4-bit Quantization + LoRA) 이해
- HuggingFace PEFT 라이브러리로 LoRA Fine-tuning 실습

## 핵심 논문
- [LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021)](https://arxiv.org/abs/2106.09685)
- [QLoRA: Efficient Finetuning of Quantized LLMs (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate peft bitsandbytes

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. PEFT 개요: 왜 전체 파라미터를 다 학습하지 않는가

### Full Fine-tuning의 문제

이전 노트북에서 봤듯이, Full Fine-tuning은 **모든 파라미터**를 업데이트한다:

| 모델 | 파라미터 수 | Full FT 메모리 |
|------|-----------|---------------|
| GPT-2 | 124M | ~1 GB |
| LLaMA-7B | 7B | ~100 GB |
| LLaMA-13B | 13B | ~180 GB |
| LLaMA-70B | 70B | ~1 TB |

### PEFT (Parameter-Efficient Fine-Tuning)

핵심 질문: **모든 파라미터를 다 바꿔야 할까?**

연구 결과, 사전학습 모델의 가중치는 **low intrinsic dimensionality**를 가진다.
즉, 전체 파라미터 공간이 아닌 **낮은 차원의 부분 공간**에서만 업데이트해도 충분하다.

### PEFT 방법들

| 방법 | 핵심 아이디어 | 학습 파라미터 |
|------|-------------|-------------|
| **LoRA** | Low-rank 행렬 추가 | 0.1~1% |
| Prefix Tuning | 입력에 학습 가능한 prefix 추가 | <0.1% |
| Prompt Tuning | 소프트 프롬프트 학습 | <0.1% |
| Adapter | 레이어 사이에 작은 네트워크 삽입 | 1~5% |
| IA3 | 활성화에 학습 가능한 벡터를 곱함 | <0.01% |

---
## 2. LoRA 수학: $W_{\text{new}} = W + BA$

### Low-Rank Decomposition

LoRA의 핵심 아이디어는 가중치 변화량 $\Delta W$를 **두 개의 작은 행렬의 곱**으로 분해하는 것이다:

$$W_{\text{new}} = W_{\text{pretrained}} + \Delta W = W + BA$$

여기서:
- $W \in \mathbb{R}^{d \times k}$: 원래 가중치 (동결, 학습 안 함)
- $B \in \mathbb{R}^{d \times r}$: LoRA down-projection
- $A \in \mathbb{R}^{r \times k}$: LoRA up-projection
- $r \ll \min(d, k)$: rank (보통 4, 8, 16, 32)

### 파라미터 절약 효과

원래 가중치 $W$의 파라미터 수: $d \times k$

LoRA로 추가되는 파라미터 수: $d \times r + r \times k = r(d + k)$

예: $d = k = 4096$, $r = 8$ 일 때:
- 원래: $4096 \times 4096 = 16,777,216$ 개
- LoRA: $8 \times (4096 + 4096) = 65,536$ 개
- **절약: 99.6%!**

### 초기화

- $A$: Kaiming uniform (또는 random Gaussian)
- $B$: **0으로 초기화** → 학습 시작 시 $\Delta W = BA = 0$
- 따라서 학습 시작 시 사전학습 모델과 동일하게 동작

In [ ]:
# LoRA 수학 직접 확인

# 원래 가중치 행렬
d, k = 4096, 4096
r = 8  # LoRA rank

W = torch.randn(d, k)  # 사전학습된 가중치 (동결)

# LoRA 행렬
B = torch.zeros(d, r)  # 0으로 초기화
A = torch.randn(r, k) * 0.01  # 작은 값으로 초기화

# Forward pass
x = torch.randn(1, k)  # 입력

# 원래 출력
original_output = x @ W.T

# LoRA 적용 출력
delta_W = B @ A  # Low-rank 변화량
lora_output = x @ (W + delta_W).T

# 시작 시에는 B=0이므로 delta_W=0, 출력 동일
print(f"원래 가중치 W shape: {W.shape}")
print(f"LoRA B shape: {B.shape}")
print(f"LoRA A shape: {A.shape}")
print(f"delta_W = B @ A shape: {delta_W.shape}")
print(f"\ndelta_W의 최대값: {delta_W.abs().max().item():.6f} (B=0이므로 0)")
print(f"original_output == lora_output? {torch.allclose(original_output, lora_output)}")

# 파라미터 수 비교
original_params = d * k
lora_params = d * r + r * k
print(f"\n원래 파라미터 수: {original_params:,}")
print(f"LoRA 파라미터 수: {lora_params:,}")
print(f"비율: {lora_params/original_params*100:.2f}% (절약: {(1-lora_params/original_params)*100:.2f}%)")

---
## 3. LoRA 시각화: 원래 행렬 vs 분해된 행렬 크기 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. 원래 가중치 행렬 W
ax = axes[0]
w_rect = mpatches.Rectangle((0, 0), 1, 1, linewidth=2, edgecolor='blue', facecolor='lightblue', alpha=0.7)
ax.add_patch(w_rect)
ax.text(0.5, 0.5, f'W\n{d} x {k}\n{d*k:,} params', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(-0.2, 1.5)
ax.set_ylim(-0.2, 1.2)
ax.set_aspect('equal')
ax.set_title('Original Weight Matrix', fontsize=13)
ax.axis('off')

# 2. LoRA 분해: B와 A
ax = axes[1]
scale = r / d  # B의 너비 비율
b_rect = mpatches.Rectangle((0, 0), scale, 1, linewidth=2, edgecolor='red', facecolor='lightyellow', alpha=0.7)
a_rect = mpatches.Rectangle((scale + 0.05, 1 - scale, 1, scale), linewidth=2, edgecolor='green', facecolor='lightgreen', alpha=0.7)
ax.add_patch(b_rect)
ax.add_patch(a_rect)
ax.text(scale/2, 0.5, f'B\n{d}x{r}', ha='center', va='center', fontsize=10, fontweight='bold', color='red')
ax.text(scale + 0.05 + 0.5, 1 - scale/2, f'A\n{r}x{k}', ha='center', va='center', fontsize=10, fontweight='bold', color='green')
ax.annotate('', xy=(scale + 0.02, 0.5), xytext=(scale + 0.05, 1 - scale),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.set_xlim(-0.2, 1.5)
ax.set_ylim(-0.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'LoRA Decomposition (r={r})', fontsize=13)
ax.axis('off')

# 3. Rank별 파라미터 수 비교
ax = axes[2]
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
ratios = [r_val * (d + k) / (d * k) * 100 for r_val in ranks]
colors = ['green' if ratio < 1 else 'orange' if ratio < 5 else 'red' for ratio in ratios]
ax.bar(range(len(ranks)), ratios, color=colors)
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([str(r_val) for r_val in ranks])
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('LoRA / Original Params (%)')
ax.set_title('Parameter Ratio by Rank', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

for i, (ratio, r_val) in enumerate(zip(ratios, ranks)):
    ax.text(i, ratio + 0.1, f'{ratio:.2f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nd={d}, k={k} 기준 Rank별 파라미터 수:")
for r_val, ratio in zip(ranks, ratios):
    lora_p = r_val * (d + k)
    print(f"  r={r_val:>3d}: {lora_p:>10,} params ({ratio:.3f}%)")

---
## 4. 어떤 레이어에 LoRA를 적용하는가

Transformer에서 LoRA를 적용할 수 있는 가중치 행렬들:

| 행렬 | 용도 | LoRA 적용 |
|------|------|----------|
| `q_proj` (Query) | Attention query 계산 | 주로 적용 |
| `k_proj` (Key) | Attention key 계산 | 가끔 적용 |
| `v_proj` (Value) | Attention value 계산 | 주로 적용 |
| `o_proj` (Output) | Attention 출력 | 가끔 적용 |
| `gate_proj` / `up_proj` | FFN 레이어 | 가끔 적용 |
| `down_proj` | FFN 레이어 | 가끔 적용 |

### 원래 LoRA 논문의 발견

- $W_q$와 $W_v$에 LoRA를 적용하는 것이 가장 효과적
- $W_k$에만 적용하면 성능이 떨어짐
- 모든 가중치에 적용할수록 성능이 좋지만 파라미터도 증가

실무에서는 **q, k, v, o 모두에 적용**하는 것이 일반적이다.

In [ ]:
# GPT-2 모델의 Attention 가중치 확인
model = AutoModelForCausalLM.from_pretrained('gpt2')

print("GPT-2 모델의 레이어 구조 (첫 번째 Transformer block):")
print("=" * 60)
for name, param in model.named_parameters():
    if 'h.0.' in name:  # 첫 번째 블록만
        print(f"  {name:>45s} | shape: {str(param.shape):>20s} | params: {param.numel():>10,}")

print(f"\n전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
del model

---
## 5. QLoRA: 4-bit Quantization + LoRA

QLoRA (Dettmers et al., 2023)는 LoRA를 더욱 메모리 효율적으로 만든 기법이다.

### 핵심 아이디어

1. **4-bit NormalFloat (NF4)**: 사전학습 가중치를 4-bit로 양자화
2. **Double Quantization**: 양자화 상수도 다시 양자화
3. **Paged Optimizers**: GPU 메모리 부족 시 CPU로 자동 오프로딩

### 메모리 절약 효과

| 방법 | 7B 모델 메모리 |
|------|---------------|
| Full Fine-tuning (fp16) | ~100 GB |
| LoRA (fp16 base) | ~14 GB |
| **QLoRA (4-bit base)** | **~6 GB** |

→ QLoRA를 사용하면 **Colab 무료 T4 GPU (16GB)**에서도 7B 모델 Fine-tuning이 가능!

In [ ]:
# Quantization에 따른 메모리 절약 시각화

precisions = ['fp32\n(32-bit)', 'fp16\n(16-bit)', 'int8\n(8-bit)', 'NF4\n(4-bit)']
bits = [32, 16, 8, 4]
memory_7b = [p / 8 * 7e9 / 1e9 for p in bits]  # 7B 모델의 메모리 (GB)

plt.figure(figsize=(8, 5))
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']
bars = plt.bar(range(len(precisions)), memory_7b, color=colors)
plt.xticks(range(len(precisions)), precisions)
plt.ylabel('Memory (GB)')
plt.title('7B Model Memory by Precision')
plt.grid(True, alpha=0.3, axis='y')

# 메모리 값 표시
for bar, mem in zip(bars, memory_7b):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{mem:.1f} GB', ha='center', fontsize=11, fontweight='bold')

# T4 GPU 메모리 라인
plt.axhline(y=16, color='gray', linestyle='--', alpha=0.7, label='T4 16GB')
plt.legend()
plt.tight_layout()
plt.show()

print("QLoRA = 4-bit 양자화 + LoRA")
print(f"→ 7B 모델이 {memory_7b[-1]:.1f} GB로 줄어들어 T4 GPU에서도 학습 가능!")

In [ ]:
# QLoRA 설정 예시 (실제 학습에 사용할 코드)
# 참고: 4-bit 양자화는 CUDA GPU가 필요합니다

print("QLoRA 설정 코드 예시:")
print("=" * 60)
print("""
from transformers import BitsAndBytesConfig

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                # 4-bit 양자화 활성화
    bnb_4bit_quant_type='nf4',       # NormalFloat4 양자화
    bnb_4bit_compute_dtype=torch.float16,  # 계산은 fp16
    bnb_4bit_use_double_quant=True,  # Double Quantization
)

# 4-bit으로 양자화된 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-hf',
    quantization_config=bnb_config,
    device_map='auto',
)

# LoRA 적용
model = get_peft_model(model, lora_config)
""")

---
## 6. 실습: HuggingFace PEFT로 LoRA Fine-tuning

GPT-2에 LoRA를 적용하여 Instruction Fine-tuning을 수행한다.

### LoraConfig 주요 파라미터

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `r` | LoRA rank | 8~64 |
| `lora_alpha` | Scaling factor ($\alpha / r$로 스케일링) | 16~32 |
| `lora_dropout` | LoRA 레이어 dropout | 0.05~0.1 |
| `target_modules` | LoRA를 적용할 레이어 이름 | 모델에 따라 다름 |
| `task_type` | 태스크 타입 | CAUSAL_LM, SEQ_CLS 등 |

In [ ]:
# Step 1: 기본 모델 로드
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

print(f"Base 모델 파라미터: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
# Step 2: LoRA 설정
lora_config = LoraConfig(
    r=8,                           # LoRA rank
    lora_alpha=16,                 # Scaling factor
    lora_dropout=0.05,             # Dropout
    target_modules=['c_attn'],     # GPT-2의 attention 가중치
    task_type=TaskType.CAUSAL_LM,  # Causal Language Modeling
    bias='none',                   # Bias는 학습하지 않음
)

print("LoRA Configuration:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  scaling (alpha/r): {lora_config.lora_alpha / lora_config.r}")
print(f"  dropout: {lora_config.lora_dropout}")
print(f"  target_modules: {lora_config.target_modules}")

In [ ]:
# Step 3: LoRA 적용
lora_model = get_peft_model(base_model, lora_config)

# 학습 가능 파라미터 확인
lora_model.print_trainable_parameters()

# 상세 확인
print("\n학습 가능한 파라미터 목록:")
print("-" * 60)
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print(f"  {name:>50s} | {str(param.shape):>15s} | {param.numel():>8,}")

In [ ]:
# Step 4: 학습 데이터 준비
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

instruction_data = [
    {"instruction": "What is machine learning?", "output": "Machine learning is a branch of AI that enables computers to learn from data without explicit programming."},
    {"instruction": "Explain overfitting.", "output": "Overfitting is when a model memorizes training data including noise, leading to poor generalization on new data."},
    {"instruction": "What is gradient descent?", "output": "Gradient descent is an optimization algorithm that adjusts parameters by moving in the direction of steepest loss decrease."},
    {"instruction": "What is a neural network?", "output": "A neural network is a computing system inspired by biological brains, consisting of interconnected layers of nodes."},
    {"instruction": "Explain transfer learning.", "output": "Transfer learning reuses a pre-trained model's knowledge for a new related task, reducing data and training requirements."},
    {"instruction": "What is regularization?", "output": "Regularization adds constraints to a model to prevent overfitting, such as L1/L2 penalties or dropout."},
    {"instruction": "What is attention mechanism?", "output": "Attention allows models to focus on relevant parts of the input when producing output, weighting importance dynamically."},
    {"instruction": "Explain the transformer architecture.", "output": "Transformers use self-attention to process sequences in parallel, enabling efficient capture of long-range dependencies."},
    {"instruction": "What is fine-tuning?", "output": "Fine-tuning adapts a pre-trained model to a specific task by training on task-specific data with a small learning rate."},
    {"instruction": "What is a loss function?", "output": "A loss function measures how well a model's predictions match the true values, guiding the optimization process."},
    {"instruction": "What is an epoch?", "output": "An epoch is one complete pass through the entire training dataset during the model training process."},
    {"instruction": "What is batch normalization?", "output": "Batch normalization normalizes layer inputs across the batch, stabilizing and accelerating neural network training."},
]

# 포맷 변환 + 반복
texts = []
for d in instruction_data:
    text = ALPACA_PROMPT.format(instruction=d['instruction'], output=d['output'])
    texts.append({'text': text})

texts = texts * 15  # 180개로 증폭
train_dataset = Dataset.from_list(texts)

def tokenize_fn(examples):
    tokenized = tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized_dataset.set_format('torch')

print(f"학습 데이터: {len(tokenized_dataset)} 개")

In [ ]:
# Step 5: LoRA 학습
training_args = TrainingArguments(
    output_dir='./gpt2-lora',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,  # LoRA는 Full FT보다 높은 LR 사용 가능
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='no',
    report_to='none',
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("LoRA 학습 시작...")
train_result = trainer.train()
print(f"\n학습 완료! Final loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 6: 결과 확인 - Full FT vs LoRA 메모리/파라미터 비교

# Loss curve
log_history = trainer.state.log_history
steps = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

if steps:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, 'r-', alpha=0.7, label='LoRA Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('LoRA Fine-tuning Loss Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 생성 테스트
lora_model.eval()

test_prompts = [
    "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is deep learning?\n\n### Response:\n",
    "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nExplain what LoRA is.\n\n### Response:\n",
]

print("LoRA Fine-tuned 모델 생성 결과:")
print("=" * 60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = lora_model.generate(
            **inputs, max_new_tokens=80, do_sample=True,
            temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    question = prompt.split('Instruction:')[1].split('Response:')[0].strip()
    answer = response.split('### Response:')[-1].strip()
    print(f"\nQ: {question}")
    print(f"A: {answer[:200]}")
    print("-" * 60)

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: LoRA Rank에 따른 성능 비교

LoRA의 rank(r)를 {2, 8, 32}로 바꿔가며 학습하고, 각각의 loss curve와 학습 가능 파라미터 수를 비교하세요.

In [ ]:
# TODO: Rank별 성능 비교
# Hint:
# ranks = [2, 8, 32]
# for r in ranks:
#     1. LoraConfig(r=r, ...) 설정
#     2. 새 base 모델 로드
#     3. get_peft_model 적용
#     4. print_trainable_parameters()로 파라미터 수 확인
#     5. Trainer로 학습 (동일한 데이터, 동일한 에폭)
#     6. loss curve 기록
# 마지막에 rank별 loss curve를 하나의 그래프에 그리기

### 연습 2: LoRA 어댑터 저장 및 로드

학습된 LoRA 어댑터를 저장하고, base 모델에 다시 로드하여 동일한 결과가 나오는지 확인하세요.

In [ ]:
# TODO: LoRA 어댑터 저장 + 로드
# Hint:
# 1. 저장: lora_model.save_pretrained('./my-lora-adapter')
# 2. 저장된 파일 크기 확인 (원래 모델보다 훨씬 작음!)
# 3. 새 base 모델 로드: base = AutoModelForCausalLM.from_pretrained('gpt2')
# 4. 어댑터 로드: model = PeftModel.from_pretrained(base, './my-lora-adapter')
# 5. 동일 프롬프트로 생성 테스트

# import os
# lora_model.save_pretrained('./my-lora-adapter')
# adapter_size = sum(os.path.getsize(os.path.join('./my-lora-adapter', f))
#                    for f in os.listdir('./my-lora-adapter'))
# print(f"LoRA 어댑터 크기: {adapter_size / 1024:.1f} KB")
# print(f"→ 원래 모델 (~500MB) 대비 매우 작음!")

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| PEFT | 일부 파라미터만 효율적으로 학습 | 메모리와 비용을 크게 절약 |
| LoRA | $W_{\text{new}} = W + BA$ (low-rank 분해) | 0.1~1% 파라미터만 학습 |
| Rank (r) | LoRA 행렬의 차원 | 높을수록 표현력 증가, 파라미터도 증가 |
| target_modules | LoRA 적용 대상 레이어 | q, v에 적용이 기본 |
| QLoRA | 4-bit 양자화 + LoRA | T4 GPU에서도 7B 모델 학습 가능 |
| 어댑터 저장 | LoRA 가중치만 저장 | 수 MB로 모델 공유 가능 |

**다음 노트북**: [04-rlhf-theory.ipynb](04-rlhf-theory.ipynb) - RLHF (Reinforcement Learning from Human Feedback)